# Laboratorium 5 (4 pkt)

Celem czwartego laboratorium jest zapoznanie się oraz zaimplementowanie algorytmów głębokiego uczenia aktywnego. Zaimplementowane algorytmy będą testowane z wykorzystaniem środowiska z OpenAI - *CartPole*.


Dołączenie standardowych bibliotek

In [1]:
from collections import deque
import gymnasium as gym
import numpy as np
import random
from copy import deepcopy

Dołączenie bibliotek do obsługi sieci neuronowych

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

torch.manual_seed(342)

class DQN(nn.Module):
    def __init__(self, state_size, action_size, hidden_neurons, num_layers=1, learning_rate=0.001):
        super(DQN, self).__init__()

        self.entrance = nn.Linear(state_size, hidden_neurons)
        for i in range(num_layers - 1):
             setattr(self, f'fc{i+1}', nn.Linear(hidden_neurons, hidden_neurons))
        self.out = nn.Linear(hidden_neurons, action_size)

        self.learning_rate = learning_rate
        self.optimizer = optim.AdamW(self.parameters(), lr=self.learning_rate)

    def forward(self, x):
        x = F.relu(self.entrance(x))
        for i in range(len(self._modules) - 2):
            x = F.relu(getattr(self, f'fc{i+1}')(x))

        return self.out(x)
    
    def predict(self, state):
        state = torch.FloatTensor(state)
        with torch.no_grad():
            q_values = self.forward(state)
        
        return q_values.numpy()
    
    def fit(self, states, targets):
        if len(states) == 0:
            return
        
        states = torch.tensor(states)
        targets = torch.tensor(targets)
        
        if states.ndim == 1:
            states = states.reshape(1, -1)
        if targets.ndim == 1:
            targets = targets.reshape(1, -1)

        states = torch.FloatTensor(states)
        targets = torch.FloatTensor(targets)

        self.optimizer.zero_grad()
        outputs = self.forward(states)
        loss = F.smooth_l1_loss(outputs, targets)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.parameters(), 1.0)
        self.optimizer.step()

## Zadanie 1 - Double Deep Q-Network

<p style='text-align: justify;'>
Celem ćwiczenie jest zaimplementowanie algorytmu Double Deep Q-Network. Wartoscią oczekiwaną sieci jest:
\begin{equation}
       Q^*(s, a) \approx r + \gamma argmax_{a'}Q_\theta'(s', a') 
\end{equation}
a wagi pomiędzy sieciami wymieniane są co dziesięć aktualizacji wag sieci sterującej poczynaniami agenta ($Q$).
</p>

In [3]:
class DDQNAgent:
    def __init__(self, state_size, action_size, model_init):
        self.state_size = state_size
        self.action_size = action_size
        self.memory = deque(maxlen=2000)
        self.gamma = 0.95    # discount rate
        self.epsilon = 0.5  # exploration rate
        self.epsilon_min = 0.01
        self.epsilon_decay = 0.95
        self.learning_rate = 0.001
        self.model = self._build_model(model_init)
        self.target_model = self._build_model(model_init)
        self.update_weights()
        self.replay_counter = 1

    def _build_model(self, model_init):
        return deepcopy(model_init)
        
    def remember(self, state, action, reward, next_state, done):
        #Function adds information to the memory about last action and its results
        self.memory.append((state, action, reward, next_state, done)) 

    def get_action(self, state):
        """
        Compute the action to take in the current state, including exploration.
        With probability self.epsilon, we should take a random action.
            otherwise - the best policy action (self.get_best_action).

        Note: To pick randomly from a list, use random.choice(list).
              To pick True or False with a given probablity, generate uniform number in [0, 1]
              and compare it with your probability
        """

        if random.uniform(0, 1) < self.epsilon:
            chosen_action = random.choice(range(self.action_size))
        else:
            chosen_action = self.get_best_action(state)
        
        return chosen_action

  
    def get_best_action(self, state):
        """
        Compute the best action to take in a state.
        """
        q_values = self.model.predict(state) 
        
        best_value = np.max(q_values)
        best_actions = np.where(q_values == best_value)[0]
        best_action = random.choice(best_actions)
        
        return best_action

    def replay(self, batch_size):
        """
        Function learn network using randomly selected actions from the memory. 
        First calculates Q value for the next state and choose action with the biggest value.
        Target value is calculated according to:
                Q(s,a) := (r + gamma * max_a(Q(s', a)))
        except the situation when the next action is the last action, in such case Q(s, a) := r.
        In order to change only those weights responsible for chosing given action, the rest values should be those
        returned by the network for state state.
        The network should be trained on batch_size samples.
        After each 10 Q Network trainings parameters should be copied to the target Q Network
        """
       
        if len(self.memory) < batch_size:
            return
        
        
        all_states = np.array([s.numpy() if torch.is_tensor(s) else s for s, a, r, ns, d in self.memory]).squeeze()
        all_next_states = np.array([ns.numpy() if torch.is_tensor(ns) else ns for s, a, r, ns, d in self.memory]).squeeze()
        all_actions = np.array([m[1] for m in self.memory])
        all_rewards = np.array([m[2] for m in self.memory])
        all_dones = np.array([m[4] for m in self.memory])

        online_q_next = self.model.predict(all_next_states)
        target_q_next = self.target_model.predict(all_next_states)

        best_actions_next = np.argmax(online_q_next, axis=1)
        
        next_q_values = target_q_next[np.arange(len(self.memory)), best_actions_next]

        all_targets = all_rewards + (self.gamma * next_q_values * (1 - all_dones))

        current_q_predictions = self.model.predict(all_states)
        q_values_of_taken_actions = current_q_predictions[np.arange(len(self.memory)), all_actions]
        
        td_errors = np.abs(q_values_of_taken_actions - all_targets)
        priorities = td_errors + 1e-6
        probabilities = priorities / np.sum(priorities)

        indices = np.random.choice(len(self.memory), batch_size, p=probabilities)
        
        states_batch = all_states[indices]
        
        targets_batch = current_q_predictions[indices].copy()
        targets_batch[np.arange(batch_size), all_actions[indices]] = all_targets[indices]

        self.model.fit(states_batch, targets_batch)
        
        self.replay_counter += 1
        if self.replay_counter % 10 == 0:
            self.update_weights()


    def update_epsilon_value(self):
        #Every each epoch epsilon value should be updated according to equation: 
        #self.epsilon *= self.epsilon_decay, but the updated value shouldn't be lower then epsilon_min value
        
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

    def update_weights(self):
        """copy trained Q Network params to target Q Network"""
        
        self.target_model = deepcopy(self.model)
        


Czas przygotować model sieci, która będzie się uczyła działania w środowisku [*CartPool*](https://gym.openai.com/envs/CartPole-v0/):

In [4]:
env = gym.make("CartPole-v0").env
state_size = env.observation_space.shape[0]
action_size = env.action_space.n
learning_rate = 0.001

print(f"State size: {state_size}, Action size: {action_size}")

model = DQN(state_size, action_size, hidden_neurons=24, num_layers=2 , learning_rate=learning_rate)

c:\Users\Filip\Documents\mgr-siium\guzw\.venv\Lib\site-packages\gymnasium\envs\registration.py:512: DeprecationWarning: WARN: The environment CartPole-v0 is out of date. You should consider upgrading to version `v1`.
  logger.deprecation(


State size: 4, Action size: 2


Czas nauczyć agenta gry w środowisku *CartPool*:

In [5]:
agent = DDQNAgent(state_size, action_size, model)

agent.epsilon = 0.6
agent.epsilon_decay = 0.8

done = False
batch_size = 64
EPISODES = 1000
counter = 0
for e in range(EPISODES):
    summary = []
    for _ in range(100):
        total_reward = 0
        env_state = env.reset()[0]
    
        state = torch.tensor(env_state, dtype=torch.float32)
        
        for time in range(500):
            action = agent.get_action(state)
            next_state_env, reward, done, _, _ = env.step(action)
            total_reward += reward

            next_state = torch.tensor(next_state_env, dtype=torch.float32)

            agent.remember(state, action, reward, next_state, done)
            state = next_state
            if done:
                break

            if len(agent.memory) > batch_size and time % 10 == 0:
                agent.replay(batch_size)

        if len(agent.memory) > batch_size:
            agent.replay(batch_size)
        
        summary.append(total_reward)

    agent.update_epsilon_value()
        
    print("epoch #{}\tmean reward = {:.3f}\tepsilon = {:.3f}".format(e, np.mean(summary), agent.epsilon))    
    
    if np.mean(summary) > 195:
        print ("You Win!")
        break


epoch #0	mean reward = 15.120	epsilon = 0.480
epoch #1	mean reward = 28.960	epsilon = 0.384
epoch #2	mean reward = 163.330	epsilon = 0.307
epoch #3	mean reward = 269.450	epsilon = 0.246
You Win!
